In [23]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [24]:
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,
)

In [25]:
class BlogState(TypedDict):

    title:str
    outline:str
    content:str
    evaluate:int

In [26]:
def create_outline(state:BlogState) -> BlogState:

    #fetch title
    title = state['title']

    #call llm gen outline
    prompt = f'Generate a detailed outline for a blog in the topic: {title}'
    outline = model.invoke(prompt).content

    #update state
    state['outline']=outline

    return state

In [27]:
def create_blog(state: BlogState) -> BlogState:
    title = state["title"]
    outline = state["outline"]

    prompt = f"Write a detailed blog on the title - {title} using the following outline\n{outline}"

    content = model.invoke(prompt).content

    state["content"] = content

    return state

In [35]:
def evaluate(state: BlogState) -> BlogState:
    outline = state["outline"]
    blog = state["content"]

    prompt = f"""
You are a professional blog reviewer.

Compare the following blog with its outline.

Outline:
{outline}

Blog:
{blog}

Rate how well the blog follows the outline.

Return ONLY a single integer between 0 and 100.
Do not return any explanation or text.
"""

    score = model.invoke(prompt).content.strip()

    try:
        state["evaluate"] = int(score)
    except ValueError:
        state["evaluate"] = 0

    return state

In [37]:
graph = StateGraph(BlogState)

# nodes
graph.add_node("create_outline",create_outline)
graph.add_node("create_blog",create_blog)
graph.add_node("evaluate",evaluate)

# edges
graph.add_edge(START,'create_outline')
graph.add_edge("create_outline",'create_blog')
graph.add_edge('create_blog','evaluate')
graph.add_edge('evaluate',END)

workflow=graph.compile()

In [38]:
initial_state={'title':'Rise of AI in india'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in india', 'outline': 'Okay, here\'s a detailed outline for a blog post on "The Rise of AI in India," designed to be comprehensive, engaging, and well-structured for a blog audience.\n\n---\n\n## Blog Title Options:\n\n*   **The AI Awakening: How India is Becoming a Global Powerhouse**\n*   **From Silicon Valley to Bengaluru: India\'s Ascent in the AI Revolution**\n*   **India\'s AI Moment: Unpacking the Drivers, Impact, and Future**\n*   **Beyond the Hype: The Real Story of AI\'s Rise in India**\n\n---\n\n## Blog Outline: The Rise of AI in India\n\n**Target Audience:** Tech enthusiasts, business leaders, policymakers, students, general public interested in India\'s technological advancements and global AI trends.\n\n**Tone:** Informative, optimistic, balanced, forward-looking, accessible.\n\n---\n\n### I. Introduction (Approx. 150-200 words)\n\n*   **A. Catchy Hook:** Start with a compelling statement about AI\'s global impact and India\'s unique position.\n    *

In [39]:
print(final_state['outline'])

Okay, here's a detailed outline for a blog post on "The Rise of AI in India," designed to be comprehensive, engaging, and well-structured for a blog audience.

---

## Blog Title Options:

*   **The AI Awakening: How India is Becoming a Global Powerhouse**
*   **From Silicon Valley to Bengaluru: India's Ascent in the AI Revolution**
*   **India's AI Moment: Unpacking the Drivers, Impact, and Future**
*   **Beyond the Hype: The Real Story of AI's Rise in India**

---

## Blog Outline: The Rise of AI in India

**Target Audience:** Tech enthusiasts, business leaders, policymakers, students, general public interested in India's technological advancements and global AI trends.

**Tone:** Informative, optimistic, balanced, forward-looking, accessible.

---

### I. Introduction (Approx. 150-200 words)

*   **A. Catchy Hook:** Start with a compelling statement about AI's global impact and India's unique position.
    *   *Example:* "The world is in the midst of an AI revolution, and while Sili

In [40]:
print(final_state['content'])

## India's AI Moment: Unpacking the Drivers, Impact, and Future

The world is in the midst of an AI revolution, a technological wave reshaping industries, economies, and daily lives at an unprecedented pace. While global tech giants and innovation hubs often grab the headlines, a quiet yet powerful transformation is unfolding in the heart of India. This isn't just about adopting cutting-edge technology; it's about a nation rapidly emerging as a significant player in the global Artificial Intelligence landscape, driven by unique demographic, economic, and strategic factors.

India's journey with AI is poised for immense impact, not only within its borders but also on the global stage. This blog post will delve into the key drivers fueling India's AI boom, explore the diverse sectors where AI is making a tangible difference, address the challenges that lie ahead, and cast a vision for the future of AI in this vibrant nation.

---

### The "Why Now?" - Key Drivers of India's AI Boom

Indi

In [41]:
print(final_state['title'])

Rise of AI in india


In [42]:
print(final_state['evaluate'])

100
